In [ ]:
!nvidia-smi

Tue Mar 24 00:21:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

This script fine-tunes Meta's LLaMA 7B on the GSM8K (Grade School Math 8K) dataset
using DeepSpeed to make it fit on a single GPU (e.g., Colab H100/A100).

Why DeepSpeed is needed here:
- LLaMA 7B has ~7 billion parameters
- Training states alone = 7B × 20 bytes = ~140GB (won't fit on any single GPU)
- With ZeRO Stage 3 + CPU offloading, optimizer states and parameters
  live on CPU RAM, so only what's actively computing stays on GPU

How to run on Google Colab:
    1. Select a GPU runtime (H100, A100, or similar)
    2. Run the install cell below
    3. Run the training script

Install dependencies (run this in a Colab cell first):
    !pip install deepspeed transformers datasets accelerate peft bitsandbytes --quiet
    !pip install flash-attn --no-build-isolation --quiet  # optional, for speed

In [ ]:
# !pip install deepspeed>=0.9.3 accelerate peft bitsandbytes --quiet
!pip install transformers==4.51.3 accelerate==1.4.0 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.1/342.1 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 58.4 MB/s eta 0:00:00


In [ ]:
import json
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

## Config

In [ ]:
MODEL_NAME = "meta-llama/Llama-2-7b-hf"
OUTPUT_DIR = "./llama7b-gsm8k-deepspeed"
MAX_SEQ_LENGTH = 512
NUM_TRAIN_EPOCHS = 1
BATCH_SIZE = 2  # per GPU micro batch size (keep small for memory)
GRADIENT_ACCUMULATION = 8  # effective batch = 2 * 8 = 16
LEARNING_RATE = 2e-4
LORA_R = 16  # LoRA rank
LORA_ALPHA = 32
LORA_DROPOUT = 0.0

## 1. DEEPSPEED CONFIG

This is the core DeepSpeed configuration.
Key settings for single GPU:
  - stage 3: partitions params, gradients, and optimizer states
  - offload_optimizer to CPU: optimizer states live on CPU RAM
  - offload_param to CPU: model parameters live on CPU, pulled to GPU when needed

This is exactly the ZeRO-Infinity concept from the talk:
  GPU does the compute, CPU/RAM stores the heavy stuff

In [ ]:
ds_config = {
    "zero_optimization": {
        "stage": 3,

        # Offload optimizer states to CPU (the 16 bytes/param from the talk)
        "offload_optimizer": {
            "device": "cpu",
            "pin_memory": True,
        },

        # Offload parameters to CPU (pulled to GPU layer-by-layer during compute)
        "offload_param": {
            "device": "cpu",
            "pin_memory": True,
        },

        # Performance tuning for Stage 3
        "overlap_comm": True,  # overlap communication with computation
        "contiguous_gradients": True,
        "sub_group_size": 1e9,
        "reduce_bucket_size": "auto",
        "stage3_prefetch_bucket_size": "auto",
        "stage3_param_persistence_threshold": "auto",
        "stage3_max_live_parameters": 1e9,
        "stage3_max_reuse_distance": 1e9,
        "stage3_gather_16bit_weights_on_model_save": True,
    },

    # Use BF16 mixed precision (H100/A100 support this)
    # This halves the memory for activations during forward/backward pass
    "bf16": {
        "enabled": True,
    },

    # Let HuggingFace Trainer control these values
    "train_batch_size": "auto",
    "train_micro_batch_size_per_gpu": "auto",
    "gradient_accumulation_steps": "auto",
    "gradient_clipping": "auto",

    "steps_per_print": 10,
    "wall_clock_breakdown": False,
}


In [ ]:
DS_CONFIG_PATH = "ds_config.json"
with open(DS_CONFIG_PATH, "w") as f:
  json.dump(ds_config, f, indent=2)

## 2. LOAD AND FORMAT THE GSM8K DATASET

GSM8K contains grade school math word problems.
Each example has a "question" and "answer" field.
We format them into a prompt template for instruction fine-tuning.

In [ ]:
def format_gsm8k(example):
    """
    Convert GSM8K examples into instruction-following format.

    Input example:
        question: "Janet has 3 apples. She buys 2 more. How many does she have?"
        answer:   "Janet starts with 3 apples...\n#### 5"

    Output format:
        "### Question:\n{question}\n\n### Answer:\n{answer}"
    """
    question = example["question"]
    answer = example["answer"]

    text = f"""### Question:
{question}

### Answer:
{answer}"""

    return {"text": text}


print("Loading GSM8K dataset...")
dataset = load_dataset("openai/gsm8k", "main")

# Format the dataset
train_dataset = dataset["train"].map(format_gsm8k, remove_columns=dataset["train"].column_names)
eval_dataset = dataset["test"].map(format_gsm8k, remove_columns=dataset["test"].column_names)

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples:  {len(eval_dataset)}")
print(f"\nExample formatted data:\n{train_dataset[0]['text'][:300]}...")

Loading GSM8K dataset...
Train samples: 7473
Eval samples:  1319

Example formatted data:
### Question:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
 
### Answer:
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and Ma...


## 3. LOAD TOKENIZER AND MODEL

We use LoRA (Low-Rank Adaptation) on top of DeepSpeed because:
  - Full fine-tuning of 7B params is expensive even with DeepSpeed
  - LoRA only trains ~0.1% of parameters (small adapter matrices)
  - Combined with DeepSpeed offloading = very memory efficient

Think of it as: DeepSpeed handles WHERE things are stored (GPU vs CPU),
and LoRA reduces HOW MUCH needs to be trained.

In [ ]:
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# LLaMA doesn't have a pad token by default — we need one for batching
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id


def tokenize_function(examples):
    """Tokenize the formatted text and create labels for causal LM training."""
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )
    # For causal LM, labels = input_ids (model predicts next token)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized


print("Tokenizing dataset...")
train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
eval_dataset = eval_dataset.map(tokenize_function, batched=True, remove_columns=["text"])


Loading tokenizer...
Tokenizing dataset...


Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [ ]:
print("\nLoading LLaMA 7B model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    # NOTE: Do NOT set device_map here — DeepSpeed handles device placement
)

# Enable gradient checkpointing to save even more memory
# (recomputes activations during backward pass instead of storing them)
model.gradient_checkpointing_enable()


Loading LLaMA 7B model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## 4. APPLY LoRA

LoRA injects small trainable matrices into the model's attention layers.
Instead of updating all 7B parameters, we only train ~18M parameters.
The rest of the model stays frozen.

In [ ]:
print("Applying LoRA adapter...")
lora_config = LoraConfig(
    r=LORA_R,              # Rank of the low-rank matrices
    lora_alpha=LORA_ALPHA,  # Scaling factor
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    # Target the attention layers (where LoRA is most effective)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Applying LoRA adapter...
trainable params: 16,777,216 || all params: 6,755,192,832 || trainable%: 0.2484


## 5. SET UP TRAINING WITH DEEPSPEED

The HuggingFace Trainer has built-in DeepSpeed support.
You just pass the deepspeed config path — Trainer handles the rest.

Under the hood, Trainer will:
  1. Call deepspeed.initialize() with your model and config
  2. Partition the model across CPU/GPU per ZeRO Stage 3
  3. Handle all the offloading/fetching during training

In [ ]:
import torch, transformers, deepspeed
print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"deepspeed: {deepspeed.__version__}")

torch: 2.10.0+cu128
transformers: 4.51.3
deepspeed: 0.18.8


In [ ]:
print("\nSetting up training arguments...")
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,

    # Batch size settings
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,

    # Optimizer and scheduler
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    # Memory optimizations
    bf16=True,                       # Use BF16 mixed precision
    gradient_checkpointing=True,     # Trade compute for memory

    # DeepSpeed — this is the key line!
    # Just point to the config file and everything works
    deepspeed=DS_CONFIG_PATH,

    # Logging and saving
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=2,

    # Misc
    remove_unused_columns=False,
    report_to="none",  # Set to "wandb" if you want W&B logging
)


Setting up training arguments...


In [ ]:
# Data collator handles padding and creating attention masks
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

# Create the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

/tmp/ipykernel_5007/2927974679.py:9: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


## 6. Start Training!

In [ ]:
!pip install mpi4py --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.5 MB/s eta 0:00:00


In [ ]:
print("\n" + "=" * 60)
print("Starting training with DeepSpeed ZeRO Stage 3 + CPU Offload")
print("=" * 60)

trainer.train()


Starting training with DeepSpeed ZeRO Stage 3 + CPU Offload


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/transformers/trainer.py:2621: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  self.lr_scheduler.step()


Epoch,Training Loss,Validation Loss
0,0.321600,0.322918


TrainOutput(global_step=467, training_loss=0.7681583852788365, metrics={'train_runtime': 11305.8775, 'train_samples_per_second': 0.661, 'train_steps_per_second': 0.041, 'total_flos': 6109632921600.0, 'train_loss': 0.7681583852788365, 'epoch': 0.9997324056729997})

In [ ]:
print("\nSaving model...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


Saving model...


('./llama7b-gsm8k-deepspeed/tokenizer_config.json',
 './llama7b-gsm8k-deepspeed/special_tokens_map.json',
 './llama7b-gsm8k-deepspeed/tokenizer.model',
 './llama7b-gsm8k-deepspeed/added_tokens.json',
 './llama7b-gsm8k-deepspeed/tokenizer.json')

## 7. TEST THE FINE-TUNED MODEL

In [ ]:
print("\n" + "=" * 60)
print("Testing the fine-tuned model")
print("=" * 60)

test_question = "Sarah has 15 stickers. She gives 4 to her brother and buys 7 more. How many stickers does Sarah have now?"

prompt = f"""### Question:
{test_question}

### Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\nPrompt:\n{prompt}")
print(f"Model response:\n{response[len(prompt):]}")

print("\nDone! Model saved to:", OUTPUT_DIR)